# Ablation Study — Loss Weights (Hybrid LSTM-GRU AE)

This notebook runs the ablation defined in `ablation_study/run_ablation.py`.

**Rules respected:**
- No existing files modified.
- No production model/weights/threshold touched.
- All outputs go to `ablation_study/results.csv` only.
- Each experiment is appended after it finishes (safe against Colab disconnects).

## 1. (Optional) Mount Google Drive
Skip this cell if you uploaded the project directly to `/content/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure repo path
Point `REPO_ROOT` to the folder containing `backend/` and `data/reduced/`.

Examples:
- If the repo lives in Drive: `REPO_ROOT = '/content/drive/MyDrive/CyberSatDetectprojct'`
- If uploaded to Colab session: `REPO_ROOT = '/content/CyberSatDetectprojct'`

In [ ]:
import os
from pathlib import Path

REPO_ROOT = '/content/drive/MyDrive/CyberSatDetectprojct'  # <-- EDIT ME if needed
RESULTS_CSV = str(Path(REPO_ROOT) / 'ablation_study' / 'results.csv')
SPLIT_FILE = str(Path(REPO_ROOT) / 'backend' / 'config' / 'data_split_qc_filtered.json')

assert Path(REPO_ROOT, 'backend', 'models', 'train_hybrid_model.py').exists(), \
    f'train_hybrid_model.py not found under {REPO_ROOT}/backend/models. Fix REPO_ROOT.'
assert Path(REPO_ROOT, 'data', 'reduced').exists(), 'data/reduced/ not found.'
assert Path(REPO_ROOT, 'data', 'attacked_v2').exists(), 'data/attacked_v2/ not found.'
assert Path(SPLIT_FILE).exists(), 'data_split_qc_filtered.json not found.'
print('Repo OK:', REPO_ROOT)
print('Clean split:', SPLIT_FILE)
print('Results CSV target:', RESULTS_CSV)

## 3. Install dependencies
Colab already has TensorFlow + NumPy. Pin TF only if your saved code expects a specific version.

In [ ]:
!pip -q install tqdm
import tensorflow as tf, numpy as np, sys
print('TF:', tf.__version__, '| NumPy:', np.__version__, '| Python:', sys.version.split()[0])
print('GPU:', tf.config.list_physical_devices('GPU'))

## 4. Run experiments one at a time (safe against disconnects)

Each experiment trains from scratch with the listed weights, evaluates on the validation set + `data/attacked_v2/`, and appends one row to `results.csv`.

**Tip:** If a session dies mid-way, just re-run this cell — already-done experiments are skipped.

In [ ]:
EXPERIMENTS_TO_RUN = ['Baseline', 'Exp-1', 'Exp-2', 'Exp-3', 'Exp-4', 'Exp-5']

import subprocess, sys
for name in EXPERIMENTS_TO_RUN:
    print(f'\n>>> Running {name} ...')
    cmd = [
        sys.executable,
        str(Path(REPO_ROOT) / 'ablation_study' / 'run_ablation.py'),
        '--repo-root', REPO_ROOT,
        '--results-csv', RESULTS_CSV,
        '--split-file', SPLIT_FILE,
        '--only', name,
    ]
    rc = subprocess.run(cmd).returncode
    if rc != 0:
        raise RuntimeError(f'Experiment {name} failed with exit code {rc}')
    print(f'<<< {name} done')

## 5. Inspect results.csv and pick the winner

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS_CSV)
display(df)
best = df.iloc[df['F1'].astype(float).idxmax()]
print('\nBest F1 combination:')
print(best.to_string())

## 6. (Optional) Run them all at once
If you prefer a single non-stop run instead of step-by-step:

In [ ]:
!python "{REPO_ROOT}/ablation_study/run_ablation.py" \
    --repo-root "{REPO_ROOT}" \
    --results-csv "{RESULTS_CSV}" \
    --split-file "{SPLIT_FILE}"